# Paper 17 · The Integrated Character Library

Build and qualify **one character voice + one accent + one emotion** as a reusable recipe. Keep the main voice-design app unchanged until this prototype proves the complete path.

**Start here:** Run all cells to activate the configuration controls. These cells only build a plan and inspect local evidence. They do not generate audio, process recordings, approve assets, install models, or change the existing app.

The controls express what we want. A dropdown label is never proof that the audio actually delivers it.

In [1]:
from pathlib import Path
import sys, json, html
import ipywidgets as w
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'services/speaker-studio/accent_pipeline.py').exists())
sys.path.insert(0, str(ROOT / 'knowledge/research/paper-17-study'))
from character_library import Intent, CHARACTERS, ACCENTS, EMOTIONS, plan, passport_template, location
PRESET = json.loads((ROOT / 'services/speaker-studio/speech-processing-preset.json').read_text())['settings']
print('Design prototype ready. Audio generation and processing remain disconnected.')

Design prototype ready. Audio generation and processing remain disconnected.


## Configure the character

Select the identity, accent and emotion independently. **Develop** is the checkpoint path for building a new combination. **Use** becomes available only after that exact profile passes qualification; direct single-pass reuse needs its own test.

First integrated target: **Doctor Doom + Indian English + Anger**. Indian accent transfer has prior evidence. Combining it with anger is the new experiment. Russian remains available for diagnostic work; do not generalize the recent Victor result into full qualification.

In [2]:
import ipywidgets as w
from IPython.display import display, HTML
import json
controls = {}
def control(name, widget):
    controls[name] = widget
    widget.layout.width = '100%'
    return widget
def row(title, help_text, fields, context=False):
    left=w.HTML('<h3 style="margin:0 0 8px">'+title+'</h3><p>'+help_text+'</p>')
    left.layout.width='28%'
    right=w.VBox(fields,layout=w.Layout(width='70%',gap='14px'))
    result=w.HBox([left,right],layout=w.Layout(width='100%',padding='24px',gap='24px',border='1px solid '+('#596168' if context else '#72598f'),margin='0 0 14px'))
    result.add_class('character-context' if context else 'character-input')
    return result
def labeled(title,widget):return w.VBox([w.HTML('<strong>'+title+'</strong>'),widget])
display(HTML('<style>.character-input{background:#211e2c;color:#eef0f2;border-radius:12px}.character-context{background:#202628;color:#eef0f2;border-radius:12px}.character-input p,.character-context p{color:#b9b4c5}.character-input input,.character-input textarea,.character-input select,.character-context input,.character-context textarea{background:#202628!important;color:#eef0f2!important;border:1px solid #465057!important;border-radius:7px!important;padding:8px!important}.character-input .widget-label{color:#eef0f2!important}</style>'))
rows=[]
rows.append(row('Who they are','Character context · not sent to the model.',[
 labeled('Role and background',control('background',w.Textarea(value='Victor von Doom is a masked scientist, sorcerer and ruler of Latveria.'))),
 labeled('Personality and motivation',control('motivation',w.Textarea(value='Proud, disciplined and fiercely self-assured.'))),
 labeled('Relationship to the listener',control('relationship',w.Text(value='Addressing a rival.')))],True))
rows.append(row('Character voice','Who is speaking. Keep the chosen identity fixed.',[
 labeled('Saved voice',control('identity',w.Dropdown(options=[('Doctor Doom','doom'),('Santa Claus','santa')],value='doom'))),
 labeled('Voice identity description',control('identity_description',w.Textarea(value='Deep masculine voice, imposing chest resonance, precise formal diction, deliberate pacing and a restrained metallic edge.')))]))
geo=w.HTML()
rows.append(row('Accent','Pronunciation from an actual recording.',[
 labeled('Accent profile',control('accent',w.Dropdown(options=[(v['label'],k) for k,v in ACCENTS.items()],value='indian'))),
 labeled('Real accent recording',control('accent_reference',w.Dropdown(options=['ASI','RRBI']))),geo]))
rows.append(row('Target emotion','What you want to hear.',[labeled('Emotion',control('emotion',w.Dropdown(options=list(EMOTIONS),value='Anger')))]))
rows.append(row('Acting example','The performance that guides the emotion.',[
 labeled('Reference recording',control('emotion_reference',w.Dropdown(options=[('Not selected — combined reference library not connected','')]))),
 labeled('Performance direction',control('direction',w.Textarea(value='Cold, controlled anger with firm emphasis and measured pauses. Preserve the selected accent and identity.'))),w.HTML('An accent recording alone does not prove the emotion. We must test both together.')]))
count=w.HTML()
rows.append(row('Words to speak','One line, up to 35 words.',[labeled('Text',control('text',w.Textarea(value='You mistake my patience for weakness.'))),count]))
rows.append(row('Generation settings','Configure each checkpoint independently.',[
 labeled('Checkpoint',control('checkpoint',w.Dropdown(options=['Actor performance','Character conversion']))),
 labeled('Seed mode',control('seed_mode',w.Dropdown(options=['Fixed','Increment','Random']))),
 w.HBox([labeled('Seed',control('seed',w.BoundedIntText(value=42,min=0,max=2147483647))),labeled('Increment',control('increment',w.BoundedIntText(value=1,min=1,max=2147483647)))]),
 w.HBox([labeled('Steps',control('steps',w.BoundedIntText(value=10,min=1,max=128))),labeled('Guidance',control('guidance',w.BoundedFloatText(value=2,min=0.1,max=8,step=0.1)))]),
 w.HTML('Configuration only. Supported engine settings will be applied when audio operations are connected.')]))
rows.append(row('Audio processing','Original stays untouched. One processed copy beside it.',[
 control('restore',w.Checkbox(value=True,description='Resemble Enhance · Restore')),
 control('polish',w.Checkbox(value=True,description='Spotify Pedalboard · Polish')),
 labeled('Polish profile',control('polish_profile',w.Dropdown(options=['Gentle','Balanced','Present']))),
 labeled('Denoise strength',control('denoise',w.BoundedFloatText(value=.1,min=0,max=1,step=.01))),
 labeled('Evaluations',control('evaluations',w.BoundedIntText(value=64,min=1,max=128))),
 labeled('Target loudness · LUFS',control('target_lufs',w.BoundedFloatText(value=-20,min=-30,max=-12))),
 w.HTML('Restore → Polish when both are enabled. Processing is not connected yet.')]))
rows.append(row('Qualification lines','Three new sentences. Judge identity, accent, emotion, words and quality.',[
 labeled('Test line 1',control('test1',w.Text(value='I offered you a choice, and you chose poorly.'))),
 labeled('Test line 2',control('test2',w.Text(value='You will explain why my instructions were ignored.'))),
 labeled('Test line 3',control('test3',w.Text(value='My patience has limits, and you have reached them.')))]))
status=w.HTML(); preview=w.HTML(); config_output=w.Output()
checkpoint_values={k:dict(seed_mode='Fixed',seed=42,increment=1,steps=10,guidance=2) for k in ['Actor performance','Character conversion']}
current_checkpoint=['Actor performance']; switching=[False]
def switch_checkpoint(change):
    if switching[0]:return
    old=current_checkpoint[0]
    checkpoint_values[old]={k:controls[k].value for k in ('seed_mode','seed','increment','steps','guidance')}
    current_checkpoint[0]=change['new']; switching[0]=True
    for k,v in checkpoint_values[change['new']].items():controls[k].value=v
    switching[0]=False
controls['checkpoint'].observe(switch_checkpoint,names='value')
def refresh(change=None):
    a=ACCENTS[controls['accent'].value]
    options=['ASI','RRBI'] if controls['accent'].value=='indian' else [a['speaker']]
    if list(controls['accent_reference'].options)!=options:controls['accent_reference'].options=options
    geo.value='<p>'+location(controls['accent'].value)+'</p><p>'+a['source']+' · '+a['speaker']+'</p>'
    n=len(controls['text'].value.split());count.value=f'{n} / 35 words'
    controls['seed'].disabled=controls['seed_mode'].value=='Random'
    controls['increment'].disabled=controls['seed_mode'].value!='Increment'
    status.value='<strong>Configuration test only · no audio generated</strong>'
    try:
        result=plan(Intent(controls['identity'].value,controls['accent'].value,controls['emotion'].value,controls['text'].value,'Develop'))
        import html
        preview.value=''.join('<h3>'+str(i+1)+'. '+html.escape(s['name'])+'</h3><p>'+html.escape(s['instruction'])+'</p>' for i,s in enumerate(result['stages']))
    except ValueError as exc:preview.value='<strong>'+str(exc)+'</strong>'
for item in controls.values():item.observe(refresh,names='value')
def inspect_configuration(_):
    checkpoint_values[current_checkpoint[0]]={k:controls[k].value for k in ('seed_mode','seed','increment','steps','guidance')}
    with config_output:
        config_output.clear_output()
        print(json.dumps({'inputs':{k:v.value for k,v in controls.items()},'checkpoints':checkpoint_values,'audio_connected':False,'qualified':False},indent=2))
inspect_button=w.Button(description='Inspect configuration',layout=w.Layout(width='220px'))
inspect_button.on_click(inspect_configuration)
display(w.VBox(rows+[status,inspect_button,config_output,w.HTML('<h2>Resulting checkpoint plan</h2>'),preview],layout=w.Layout(width='100%')))
refresh()


## One vertical process, horizontal audio development

| Checkpoint | Original | Processed output | What moves downward |
|---|---|---|---|
| Real performance reference | Untouched recording, source metadata and exact transcript | Optional Resemble → Pedalboard output | The version the user selects |
| Actor saying the requested words | Raw generated actor take | One optional processed output | The exact accepted actor asset |
| Character conversion | Raw conversion into the pinned identity | One optional processed output | The exact accepted character take |

At generated checkpoints the main actions remain **Regenerate** and **Accept & continue**. Regenerate uses a new seed for that checkpoint only. Reprocessing replaces the one visible processed output, never the original. Replaced assets remain traceable if another result depends on them.

Changing an upstream selection makes downstream acceptance stale. Existing audio is preserved with its real lineage. A processed input must never be relabeled Original in a later attempt; the original source and the frozen generation input are separate fields.

## How accent and emotion meet

A performance recording carries pronunciation, emotion, timing and a speaker identity together. We cannot assume that attaching an accent file and an emotion file will combine those properties correctly.

**Preferred development route:** select a real performance that demonstrates both the chosen accent and emotion, generate the new words, approve that actor, then convert the accepted performance into the character.

**Experimental route if no matched performance exists:** test whether the actor model can preserve the real accent reference while applying an emotion direction. Review the actor before conversion. If it stays neutral or loses the accent, stop at that checkpoint; change the reference or supported generation method. Do not compensate by changing the final character identity.

Neither route is yet qualified for the default Indian-English angry Doom combination. We will compare the actor against its reference and Doom against that exact actor. A word check alone never approves emotion, accent or identity.

## Processing defaults

Restore and Polish are independently enabled. The current **Speech clarity** recipe is a starting configuration, not a universal optimum. Adjust it when listening exposes artifacts. Always process a copy of the stage's original; applying a new recipe must not silently compound the previous effects.

In [3]:
display(HTML('<table>'+''.join('<tr><th style="text-align:left;padding:5px">'+html.escape(k)+'</th><td>'+html.escape(str(PRESET[k]))+'</td></tr>' for k in ['preset','restore','polish','evaluations','solver','denoise','temperature','profile','target_lufs','true_peak_db','sample_rate'])+'</table>'))

preset,speech-clarity-v1
restore,True
polish,True
evaluations,64
solver,midpoint
denoise,0.1
temperature,0.5
profile,Gentle
target_lufs,-20.0
true_peak_db,-2.0
sample_rate,44100


## What is already supported, and what is new

| Evidence | Meaning |
|---|---|
| Santa anger: three lines passed user listening | Evidence for that saved voice and anger recipe only |
| Real British, American and Indian references transferred into Doom | Accent-only evidence for the tested recordings and lines |
| Russian preprocessing helped the Victor opening | Useful local diagnostic result; still inspect accent and final quality |
| Speech clarity processing ran and passed technical checks | Valid audio output; not proof that every voice sounds better |
| Accent + emotion together | **New work. Not qualified yet** |
| A saved combined profile generating directly in one pass | **Separate optimization test. Not qualified yet** |

Source locations remain optional. For ASI, Hindi is documented as native language; country and locality remain unknown rather than being guessed.

## The qualification gate

Use three fresh sentences, each at most 35 words, with the same pinned recipe. For **each** sentence record five listening verdicts: correct words, same character, chosen accent, chosen emotion, acceptable audio quality. All five must pass on all three lines. Automatic transcription and audio-integrity checks support this decision but do not replace listening.

Regenerating one failure does not erase it from the passport. Save a profile only when its accepted assets, model versions, per-stage seeds, processing parameters and qualification evidence are complete.

In [4]:
# Executable contract checks; no audio engines are called.
from character_library import qualify
verdict = dict(words=True, identity=True, accent=True, emotion=True, quality=True)
assert qualify([verdict, verdict, verdict])
assert not qualify([verdict, verdict, dict(verdict, accent=False)])
assert not plan(Intent(mode='Use'))['ready_for_future_generation']
assert not plan(Intent())['can_generate']
print('Qualification contracts pass. No voice profile has been approved by these checks.')

Qualification contracts pass. No voice profile has been approved by these checks.


## Prototype build order

1. **This notebook:** agree on the input contract, visible checkpoints, immutable assets and acceptance rules. Configuration controls and qualification rules are executable; generation is not connected yet.
2. **Connect one combination:** Doctor Doom + Indian English + Anger. Reuse the existing reference catalog, processing worker, actor generation and conversion adapters. Add the missing combined-performance trial and capture every intermediate recording.
3. **Qualify three new lines:** record the user's five listening verdicts per line. Keep an explicit failure path to the weak checkpoint.
4. **Save and reuse the profile:** new text runs the verified recipe without repeating setup. Keep the checkpoint view accessible. Test direct single-pass generation separately before enabling it.
5. **Upgrade the app:** reuse the same library/profile service and contracts behind the app's Character / Accent / Emotion selectors. Do not build an independent second pipeline.

**Promotion decision:** no app integration until the integrated notebook path passes the combined listening test and survives save/reopen with original, selected input, results and passport intact.

## Existing evidence and implementation anchors

The design is grounded in the current repository, not a new model recommendation:

- Paper 16 and `knowledge/research/paper-16-study/` contain accent, emotion and processing evidence.
- `services/speaker-studio/accent_pipeline.py` holds durable checkpoints, asset integrity, explicit acceptance and independent retries.
- `services/speaker-studio/accent_processing.py` connects the existing local Resemble Enhance and Spotify Pedalboard worker.
- `speech-processing-preset.json` supplies the adjustable Speech clarity default.
- `original-reference-repair-verification.json` records the correction separating canonical raw references from frozen generation inputs.
- `paper-17-study/character_library.py` is this notebook's design-only planner and passport template. It does not call audio engines.

The current app and accent listening workspace remain separate and available. No services are stopped or replaced by running this notebook.